# Variant 7 — Answer-only LoRA + Rule Solver Fallback

Notebook này được tạo lại từ hướng variant đã chọn:

- Fine-tune `NlpHUST/gpt2-vietnamese` theo dạng **answer-only**.
- Không ép GPT-2 sinh lời giải dài trong target.
- Dùng rule solver đơn giản trước model cho các template chắc chắn.
- Khi ghi prediction, luôn wrap về format: `Lời giải ngắn: ...
Đáp án là: <answer>`.
- Sinh `valid_output.json`, `valid_report.json`, và nếu có test thì sinh `/kaggle/working/test_predictions.json`.

> Chạy Kaggle: Internet OFF, GPU ON.

## Cell 1 — Setup, config, paths

In [1]:
import os
import re
import gc
import json
import math
import random
import inspect
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

VARIANT_NAME = "v9_solution_full_preprocess_lora_rule_solver"
SAFE_EOS_ID = 50256

# Kaggle official paths. Local fallback paths are included only for debugging outside Kaggle.
TRAIN_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/dataset-math/train.json"),
    Path("/kaggle/input/dataset-math/train.json"),
    Path("/mnt/data/train.json"),
]
VALID_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/dataset-math/valid.json"),
    Path("/kaggle/input/dataset-math/valid.json"),
    Path("/mnt/data/valid.json"),
]
TEST_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/dataset-math/test.json"),
    Path("/kaggle/input/dataset-math/test.json"),
    Path("/mnt/data/test.json"),
]
MODEL_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese"),
    Path("/kaggle/input/nlphustgpt2-vietnamese"),
    Path("/kaggle/input/nlp-hust-gpt2-vietnamese"),
    Path("/kaggle/input/gpt2-vietnamese"),
]

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")
PROC_DIR = WORK_DIR / f"processed_{VARIANT_NAME}"
MODEL_OUT_DIR = WORK_DIR / f"lora_{VARIANT_NAME}"
PRED_DIR = WORK_DIR / f"predictions_{VARIANT_NAME}"
for p in [PROC_DIR, MODEL_OUT_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Main knobs for <= 3 hours. Reduce TRAIN_MAX_SAMPLES / NUM_TRAIN_EPOCHS if your Kaggle GPU is slow.
TRAIN_MAX_SAMPLES = 40000
MAX_LENGTH = 512 # 380 cho answer_only
GEN_MAX_NEW_TOKENS = 160 # 20 cho answer_only

NUM_TRAIN_EPOCHS = 4
PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

LOGGING_STEPS = 50
EVAL_STEPS = 700
SAVE_STEPS = 700
EVAL_LOSS_MAX_SAMPLES = 512
VALID_EVAL_MAX_SAMPLES = None  # None = full valid; set 300/500 for quick smoke test.

DO_TRAIN = True
DO_VALIDATE = True
DO_TEST_PREDICT = True

print("Variant:", VARIANT_NAME)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Variant: v9_solution_full_preprocess_lora_rule_solver
CUDA: True
GPU: Tesla T4


## Cell 2 — I/O helpers and path discovery

In [2]:
def first_existing(paths: List[Path], required: bool = True, name: str = "path") -> Optional[Path]:
    for p in paths:
        if p.exists():
            print(f"{name}: {p}")
            return p
    if required:
        raise FileNotFoundError(f"Cannot find {name}. Tried: {[str(p) for p in paths]}")
    print(f"{name}: not found")
    return None

TRAIN_PATH = first_existing(TRAIN_PATH_CANDIDATES, True, "TRAIN_PATH")
VALID_PATH = first_existing(VALID_PATH_CANDIDATES, True, "VALID_PATH")
TEST_PATH = first_existing(TEST_PATH_CANDIDATES, False, "TEST_PATH")


def find_model_path() -> Path:
    for p in MODEL_PATH_CANDIDATES:
        if p.exists():
            print("MODEL_PATH:", p)
            return p
    # Fallback: search common Kaggle input subdirs for config.json.
    root = Path("/kaggle/input")
    if root.exists():
        for cfg in root.rglob("config.json"):
            parent = cfg.parent
            files = {x.name for x in parent.iterdir() if x.is_file()}
            if any(name.startswith("pytorch_model") or name.endswith(".safetensors") for name in files):
                print("MODEL_PATH auto-found:", parent)
                return parent
    raise FileNotFoundError("Cannot find local NlpHUST/gpt2-vietnamese model directory.")


def read_json_or_jsonl(path: Path) -> Any:
    text = path.read_text(encoding="utf-8-sig").strip()
    if not text:
        return []
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # JSONL fallback.
    records = []
    ok_jsonl = True
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            ok_jsonl = False
            break
    if ok_jsonl and records:
        return records

    # Concatenated JSON objects fallback.
    decoder = json.JSONDecoder()
    idx = 0
    records = []
    while idx < len(text):
        while idx < len(text) and text[idx].isspace():
            idx += 1
        if idx >= len(text):
            break
        obj, end = decoder.raw_decode(text, idx)
        records.append(obj)
        idx = end
    return records


def ensure_list_records(obj: Any) -> List[Dict[str, Any]]:
    if isinstance(obj, list):
        return [x for x in obj if isinstance(x, dict)]
    if isinstance(obj, dict):
        for key in ["data", "records", "items", "examples"]:
            if isinstance(obj.get(key), list):
                return [x for x in obj[key] if isinstance(x, dict)]
        return [obj]
    return []


def write_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def write_jsonl(records: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

raw_train = ensure_list_records(read_json_or_jsonl(TRAIN_PATH))
raw_valid = ensure_list_records(read_json_or_jsonl(VALID_PATH))
raw_test = ensure_list_records(read_json_or_jsonl(TEST_PATH)) if TEST_PATH is not None and TEST_PATH.exists() else []

MODEL_PATH = find_model_path()

print("raw_train:", len(raw_train), "raw_valid:", len(raw_valid), "raw_test:", len(raw_test))
print("train sample keys:", raw_train[0].keys() if raw_train else None)

TRAIN_PATH: /kaggle/input/datasets/kimanh2002/dataset-math/train.json
VALID_PATH: /kaggle/input/datasets/kimanh2002/dataset-math/valid.json
TEST_PATH: not found
MODEL_PATH: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
raw_train: 95400 raw_valid: 1000 raw_test: 0
train sample keys: dict_keys(['original_question_vi', 'original_question_en', 'query_vi', 'query_en', 'response_vi', 'response_en', 'type'])


## Cell 3 — Answer extraction and scoring

Extractor này ưu tiên đáp án sau anchor (`Đáp án là`, `Câu trả lời là`, `The answer is`, `####`) và xử lý cơ bản `\frac{a}{b}`, `a/b`, `a\pi`, decimal comma, thousand separator.

In [3]:
ANSWER_ANCHOR_PATTERNS = [
    r"Đáp án là\s*[:：]?\s*([^\n]+)",
    r"Câu trả lời là\s*[:：]?\s*([^\n]+)",
    r"The answer is\s*[:：]?\s*([^\n]+)",
    r"####\s*([^\n]+)",
]

BAD_ANSWER_TOKENS = ["nan", "inf", "undefined", "không xác định", "không đủ", "unknown", "none", "n/a"]


def normalize_unicode_text(text: str) -> str:
    text = str(text or "")
    text = text.replace("\ufeff", "").replace("\u200b", "").replace("\u00a0", " ")
    text = text.replace("−", "-").replace("–", "-").replace("—", "-")
    text = text.replace("×", "*").replace("÷", "/")
    text = text.replace("\\đóng hộp", "\\boxed").replace("\\ đóng hộp", "\\boxed")
    text = text.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    return text


def strip_asy_blocks(text: str) -> str:
    return re.sub(r"\[asy\].*?\[/asy\]", " ", str(text or ""), flags=re.I | re.S)


def unwrap_boxed_once(s: str) -> str:
    m = re.search(r"\\boxed\s*\{", s)
    if not m:
        return s
    start = m.end()
    depth = 1
    i = start
    while i < len(s):
        if s[i] == "{":
            depth += 1
        elif s[i] == "}":
            depth -= 1
            if depth == 0:
                return s[:m.start()] + s[start:i] + s[i+1:]
        i += 1
    return s


def strip_boxed(s: str) -> str:
    s = str(s or "")
    for _ in range(5):
        s2 = unwrap_boxed_once(s)
        if s2 == s:
            break
        s = s2
    return s


def clean_answer_candidate(ans: str) -> str:
    ans = normalize_unicode_text(ans)
    ans = strip_boxed(ans).strip()
    ans = ans.replace("$", "").replace("`", "").strip()
    ans = re.split(r"\s+(?:Đáp án là|Câu trả lời là|The answer is)\s*[:：]?", ans, flags=re.I)[0]
    ans = ans.split("\n")[0].strip()
    ans = re.sub(r"^[=:：\s]+", "", ans).strip()
    ans = re.sub(r"(?:\.|,|;|:|。)+$", "", ans).strip()
    ans = re.sub(r"\s+", " ", ans)

    # Prefer latex fraction if present.
    m = re.search(r"-?\\frac\s*\{\s*-?\d+(?:[\.,]\d+)?\s*\}\s*\{\s*-?\d+(?:[\.,]\d+)?\s*\}", ans)
    if m:
        return m.group(0).replace(" ", "")

    # Prefer tuple if present.
    m = re.search(r"\((-?\d+(?:[\.,]\d+)?\s*,\s*)+-?\d+(?:[\.,]\d+)?\)", ans)
    if m:
        return re.sub(r"\s+", "", m.group(0))

    # Prefer coefficient*pi form.
    m = re.search(r"-?\d+(?:[\.,]\d+)?\s*\\?pi", ans, flags=re.I)
    if m:
        return m.group(0).replace(" ", "")

    # Prefer common fraction.
    m = re.search(r"-?\d+(?:[\.,]\d+)?\s*/\s*-?\d+(?:[\.,]\d+)?", ans)
    if m:
        return m.group(0).replace(" ", "")

    # Otherwise last numeric token, with optional thousand separators.
    nums = re.findall(r"-?\d+(?:[\.,]\d+)*", ans)
    if nums:
        return nums[-1]
    return ans.strip()


def extract_final_answer(text: str, fallback_last_number: bool = True) -> Optional[str]:
    text = normalize_unicode_text(text)
    if not text.strip():
        return None
    matches = []
    for pat in ANSWER_ANCHOR_PATTERNS:
        for m in re.finditer(pat, text, flags=re.I):
            matches.append((m.start(), m.group(1)))
    if matches:
        matches.sort(key=lambda x: x[0])
        cand = clean_answer_candidate(matches[-1][1])
        if cand and not any(tok in cand.lower() for tok in BAD_ANSWER_TOKENS):
            return cand
    if fallback_last_number:
        cand = clean_answer_candidate(text)
        if cand and not any(tok in cand.lower() for tok in BAD_ANSWER_TOKENS):
            return cand
    return None


def _normalize_num_token(x: str) -> str:
    x = str(x).strip().replace(" ", "")
    # 40.320 or 1,234,567 => thousand separators.
    if re.fullmatch(r"-?\d{1,3}([\.,]\d{3})+", x):
        return x.replace(".", "").replace(",", "")
    # 4,5 => 4.5 decimal comma.
    if re.fullmatch(r"-?\d+,\d{1,6}", x):
        return x.replace(",", ".")
    return x.replace(",", "")


def parse_numeric_answer(ans: Any) -> Optional[float]:
    if ans is None:
        return None
    s = clean_answer_candidate(str(ans))
    s = normalize_unicode_text(s)
    s = s.replace("$", "").strip()

    # Latex fraction.
    m = re.fullmatch(r"(-?)\\frac\{\s*(-?\d+(?:[\.,]\d+)?)\s*\}\{\s*(-?\d+(?:[\.,]\d+)?)\s*\}", s)
    if m:
        sign = -1.0 if m.group(1) == "-" else 1.0
        a = float(_normalize_num_token(m.group(2)))
        b = float(_normalize_num_token(m.group(3)))
        return sign * a / b if b != 0 else None

    # Common fraction.
    m = re.fullmatch(r"(-?\d+(?:[\.,]\d+)?)\s*/\s*(-?\d+(?:[\.,]\d+)?)", s)
    if m:
        a = float(_normalize_num_token(m.group(1)))
        b = float(_normalize_num_token(m.group(2)))
        return a / b if b != 0 else None

    # pi forms: pi, -pi/2, 36\pi.
    pi_s = s.replace("\\pi", "pi").replace("π", "pi")
    m = re.fullmatch(r"(-?)(?:(\d+(?:[\.,]\d+)?))?\s*pi(?:\s*/\s*(\d+(?:[\.,]\d+)?))?", pi_s, flags=re.I)
    if m:
        sign = -1.0 if m.group(1) == "-" else 1.0
        coef = float(_normalize_num_token(m.group(2))) if m.group(2) else 1.0
        den = float(_normalize_num_token(m.group(3))) if m.group(3) else 1.0
        return sign * coef * math.pi / den if den != 0 else None

    # Plain number.
    if re.fullmatch(r"-?\d+(?:[\.,]\d+)*", s):
        try:
            return float(_normalize_num_token(s))
        except Exception:
            return None
    return None


def relative_error(pred: Any, gold: Any) -> Optional[float]:
    p = parse_numeric_answer(pred)
    g = parse_numeric_answer(gold)
    if p is None or g is None:
        return None
    return abs(p - g) / max(1.0, abs(g))


def score_one(pred: Any, gold: Any) -> int:
    # Exact string fallback for non-scalar answers.
    if pred is not None and gold is not None:
        if clean_answer_candidate(str(pred)) == clean_answer_candidate(str(gold)):
            return 10
    err = relative_error(pred, gold)
    if err is None:
        return 0
    if err <= 0.01:
        return 10
    if err <= 0.10:
        return 5
    if err <= 0.50:
        return 1
    return 0

# Quick sanity checks.
for s in ["Đáp án là: 37", "Câu trả lời là: \\frac{9}{20}", "The answer is: 36\\pi", "#### 40.320"]:
    a = extract_final_answer(s)
    print(s, "=>", a, "=>", parse_numeric_answer(a))


Đáp án là: 37 => 37 => 37.0
Câu trả lời là: \frac{9}{20} => \frac{9}{20} => 0.45
The answer is: 36\pi => 36\pi => 113.09733552923255
#### 40.320 => 40.320 => 40320.0


## Cell 3b — Pipeline Preprocessing

In [4]:
import re, json, math, hashlib
from pathlib import Path
from collections import Counter, defaultdict
from transformers import AutoTokenizer
import numpy as np

# ── Paths: notebook-compatible ─────────────────────────────────────────────────
# Yêu cầu: Cell 2 + Cell 4 đã chạy trước đó, có WORK_DIR, PROC_DIR, MODEL_PATH, MAX_LENGTH.
PROJECT_ROOT = WORK_DIR

# Nếu dùng trực tiếp full_pipeline(raw_train, raw_valid, raw_test)
# thì TRAIN_FILE/VALID_FILE/TEST_FILE không quá quan trọng.
# Nhưng vẫn set để load_raw_data() không bị lệch nếu cần dùng.
DATA_DIR = PROJECT_ROOT / "dataset"
TRAIN_FILE = TRAIN_PATH
VALID_FILE = VALID_PATH
TEST_FILE = TEST_PATH if (TEST_PATH is not None and TEST_PATH.exists()) else None

GENERATED_DATA_DIR = PROC_DIR

PREPROCESSED_TRAIN_FILE = PROC_DIR / "train_preprocessed_answer_only.jsonl"
PREPROCESSED_VALID_FILE = PROC_DIR / "valid_preprocessed_answer_only.jsonl"
PREPROCESSED_TEST_FILE = PROC_DIR / "test_preprocessed_answer_only.jsonl"
PREPROCESSING_REPORT_FILE = PROC_DIR / "train_preprocessing_report.json"

# Tokenizer/model path phải lấy từ notebook, không dùng PROJECT_ROOT / "NlpHUST-gpt2-vietnamese"
MODEL_DIR = MODEL_PATH
TOKENIZER_LOCAL_ONLY = True

# ── Config: answer-only mode for current notebook ──────────────────────────────
DROP_TRAIN_WITHOUT_FINAL_ANSWER = True
MAX_QUERY_CHARS = 600
MAX_QUERY_WORDS = 130

# Giữ notebook theo hướng answer_only / solution
TARGET_MODE = "solution"

DROP_ENGLISH_DOMINANT = True
DROP_BAD_ANSWER_FORMAT = True
DROP_CONFLICT_GROUPS = True

# Với answer-only, response_vi sau preprocess chỉ còn "Đáp án là: ..."
# nên truncation gần như không còn quan trọng.
MAX_RESPONSE_WORDS = 512
TRUNCATION_MARKER = "\n...\n"

# Nếu để 20, có thể drop một số bài rất ngắn nhưng hợp lệ.
# Với answer-only notebook, mình khuyên dùng 8.
QUALITY_FILTER_MIN_WORDS = 8
QUALITY_FILTER_MAX_WORDS = 450

ARITH_FILTER_ENABLED = False
ARITH_MAX_FAIL_RATIO = 0.70

# ── Token audit: align with notebook MAX_LENGTH ────────────────────────────────
TOKEN_AUDIT_ENABLED = True
TOKEN_AUDIT_DROP_OVER_LIMIT = True

# Notebook đang train max_length=256, nên preprocessing audit cũng phải là 256.
MAX_TRAIN_TOKENS = MAX_LENGTH

TOKEN_AUDIT_PREVIEW_LIMIT = 20

# Giữ tối thiểu mỗi type để tránh mất quá nhiều sample nhóm hiếm.
TOKEN_MIN_KEEP_PER_TYPE = 1500
TOKEN_DROP_PROTECT_RARE_TYPES = True

# ── Module-level compiled regex ────────────────────────────────────────────────
ARITH_EQ_RE = re.compile(
    r"([-+]?\d+(?:[.,]\d+)?)\s*([+\-*/×÷])\s*([-+]?\d+(?:[.,]\d+)?)\s*=\s*([-+]?\d+(?:[.,]\d+)?)"
)
ANSWER_ANCHORS = [
    r"Đáp\s*án\s*là",
    r"Câu\s*trả\s*lời\s*là",
    r"(?:Câu\s+)?Trả\s*lời(?:\s+là)?",
    r"Đáp\s*án(?:\s+[A-Za-z]{1,4}\d{0,2})?",
    r"Kết\s*quả\s*là",
    r"Vậy\s*đáp\s*án\s*là",
    r"The answer is",
    r"Answer",
    r"####",
]
ANSWER_ANCHOR_RE = re.compile(
    r"(?:"
    + "|".join(ANSWER_ANCHORS)
    + r")\s*[:：]?",
    flags=re.IGNORECASE,
)
ANSWER_CLEAN_RE = re.compile(
    r"(?:^|\n)\s*(?:"
    + "|".join(ANSWER_ANCHORS)
    + r")\s*[:：]?\s*[^\n]*\s*$",
    flags=re.IGNORECASE,
)
NUM_RE = re.compile(r"[-+]?\d[\d.,]*(?:\s*/\s*[-+]?\d[\d.,]*)?")
PLAIN_NUMERIC_RE = re.compile(r"[-+]?\d[\d.,]*(?:\s*/\s*[-+]?\d[\d.,]*)?")
VI_DIACRITIC_RE = re.compile(
    r"[ăâđêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ]",
    flags=re.IGNORECASE,
)
VI_HINT = {
    "một", "là", "có", "của", "trong", "cho", "nếu", "tìm", "tính", "hãy", "bao", "nhiêu", 
    "giá", "trị", "biến", "số", "bằng", "được", "người", "học", "sinh", "quả", "chiếc", "cái",
}
EN_HINT = {"the","and","if","what","how","many","much","find","calculate","total","given",
           "is","are","has","have","wants","wanted","people","apples","number","value",
           "answer","equation","solve","there","were","will"}
TYPE_BONUS = {
    "GSM_AnsAug":50,"GSM_Rephrased":45,"MATH_AnsAug":25,"MATH_Rephrased":20,
    "GSM_FOBAR":10,"GSM_SV":10,"MATH_FOBAR":5,"MATH_SV":5,
}


# ── Helpers ────────────────────────────────────────────────────────────────────
def normalize_space(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()

def word_count(text):
    return len(re.findall(r"\S+", str(text or "")))

def save_json(obj, path):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def save_jsonl(records, path):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in records) + "\n", encoding="utf-8")

def fingerprint(records):
    content = json.dumps(
        [r.get("query_vi", "") + "\n" + r.get("response_vi", "") for r in records],
        ensure_ascii=False,
        sort_keys=True,
    ).encode("utf-8")
    return hashlib.md5(content).hexdigest()

def _dedup_report(records, mode):
    before = Counter(r.get("type","unknown") for r in records)
    after = Counter(r.get("type","unknown") for r in records)
    return {"mode": mode, "before_by_type": dict(before), "after_by_type": dict(after),
            "kept_ratio": {t: round(after[t]/max(1,before[t]),4) for t in before}}

def build_train_prompt(rec):
    q = str(rec.get("query_vi", "")).strip()
    if TARGET_MODE == "answer_only": return f"Bài toán: {q}\n\n"
    return f"Bài toán: {q}\n\nLời giải:\n"

def build_train_text(rec):
    return build_train_prompt(rec) + str(rec.get("response_vi", "")).strip()

# ── Load data ─────────────────────────────────────────────────────────────────
def load_records(path, need_response=False):
    p = Path(path)
    with p.open("r", encoding="utf-8-sig") as f:
        first = f.read(1); f.seek(0)
        records = json.load(f) if first == "[" else [json.loads(l) for l in f if l.strip()]
    out = []
    for i, rec in enumerate(records):
        if "query_vi" not in rec:
            raise KeyError(f"Dong {i} thiếu query_vi")
        if need_response and "response_vi" not in rec:
            raise KeyError(f"Dong {i} thiếu response_vi")
        item = dict(rec); item.setdefault("id", i); item.setdefault("type","unknown")
        out.append(item)
    return out

def load_raw_data():
    raw_train = load_records(TRAIN_FILE, True)
    raw_valid = load_records(VALID_FILE, True) if VALID_FILE.exists() else []
    raw_test = load_records(TEST_FILE) if TEST_FILE else []
    return raw_train, raw_valid, raw_test

def load_audit_tokenizer():
    try:
        tok = AutoTokenizer.from_pretrained(
            str(MODEL_DIR), local_files_only=TOKENIZER_LOCAL_ONLY,
        )
        if getattr(tok, "pad_token_id", None) is None and getattr(tok, "eos_token", None) is not None:
            tok.pad_token = tok.eos_token
        return tok
    except Exception as e:
        print(f"[WARN] Token audit skipped: cannot load tokenizer from {MODEL_DIR}. Error: {e}")
        return None


# ── Text normalization ─────────────────────────────────────────────────────────
def fix_artifacts(text):
    text = str(text or "")
    text = re.sub(r"\\\s*đóng\s*hộp\s*\{", r"\\boxed{", text, flags=re.IGNORECASE)
    text = re.sub(r"\\\s*boxed\s*\{", r"\\boxed{", text, flags=re.IGNORECASE)
    text = text.replace("\u200b", "").replace("\ufeff", "")
    return text.strip()

def strip_asy(text):
    text = str(text or "")
    text = re.sub(r"\[asy\].*?\[/asy\]", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"\[asy\].*?(?=(?:Giá trị của|Giá trị là|Câu trả lời|Đáp án|Nếu chúng ta biết|Để giải|$))", "", text, flags=re.DOTALL | re.IGNORECASE,)
    text = re.sub(r"\[/asy\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()

def clean_text(text):
    text = str(text or "")
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"(Giá trị của biến [^\n?]+\?)\s*\1", r"\1", text)
    if re.search(r"Đáp án là|Câu trả lời là|####|\\boxed", text, flags=re.IGNORECASE):
        text = re.sub(r"\n(?:The answer is[:\s]+[\d.,/\\{}a-zA-Z]+\.?\s*)+$", "", text, flags=re.IGNORECASE)
    return text.strip()


# ── Brace parser ───────────────────────────────────────────────────────────────
def parse_braces(text, start):
    if start >= len(text) or text[start] != "{":
        return None, -1
    depth, i, content_start = 0, start, start + 1
    while i < len(text):
        if text[i] == "{": depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[content_start:i], i
        i += 1
    return None, -1

def extract_boxed(text):
    text = str(text or ""); idx = text.find(r"\boxed")
    return parse_braces(text, idx + 6)[0] if idx != -1 else None


# ── Answer extraction ───────────────────────────────────────────────────────────
def _first_answer_unit(text):
    if not text:
        return None
    boxed = extract_boxed(text)
    if boxed is not None:
        return boxed.strip()
    first_line = text.split("\n", 1)[0].strip()
    if first_line:
        first_line = re.sub(r"^(?:là|=|:|：)\s*", "", first_line, flags=re.IGNORECASE)
        if first_line:
            return first_line.strip()
    nums = NUM_RE.findall(text)
    return nums[-1].strip() if nums else None

def _last_answer_unit(text):
    text = str(text or ""); boxes, pos = [], 0
    while True:
        idx = text.find(r"\boxed", pos)
        if idx == -1: break
        c = parse_braces(text, idx + 6)[0]
        if c is not None: boxes.append(c)
        pos = idx + 1
    if boxes: return boxes[-1].strip()
    nums = NUM_RE.findall(text)
    return nums[-1].strip() if nums else None

def extract_answer(text, allow_last=False, prefer_first=False):
    text = str(text or "")
    matches = list(ANSWER_ANCHOR_RE.finditer(text))
    if matches:
        m = matches[0] if prefer_first else matches[-1]
        ans = _first_answer_unit(text[m.end():])
        if ans is not None: return ans
    return _last_answer_unit(text) if allow_last else None

def clean_answer_tail(text):
    if text is None: return None
    text = str(text).strip().split("\n", 1)[0]
    text = re.sub(r"^(?:là|=|:|：)\s*", "", text, flags=re.IGNORECASE)
    return text.strip(" .。;；,，") or None

def rebuild_response(response, answer):
    cleaned = str(response or "").strip()

    cleaned = re.sub(
        r"\s*(?:Đáp án là|Câu trả lời là|(?:Câu\s+)?Trả lời(?:\s+là)?|Đáp án|The answer is|Answer)\s*[:：]?\s*[^\n]*\s*$",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )

    cleaned = re.sub(
        r"\s*####\s*[^\n]*\s*$",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )

    lines = cleaned.split("\n")
    if lines and re.match(r"^\s*\\boxed\s*\{.*\}\s*[.。]?\s*$", lines[-1].strip()):
        lines = lines[:-1]

    cleaned = "\n".join(lines).rstrip()
    return (cleaned + f"\nĐáp án là: {answer}").strip()

# ── Number parsing ──────────────────────────────────────────────────────────────
def _parse_number_core(text):
    """Parse mot plain number/fraction string -> float. Khong parse LaTeX/symbolic."""
    text = str(text).strip().replace(" ", "")
    if not text or re.search(r"\d{16,}", text.replace(",","").replace(".","")):
        return None
    if "/" in text:
        parts = text.split("/")
        if len(parts) == 2:
            a, b = _parse_number_core(parts[0]), _parse_number_core(parts[1])
            if a is not None and b not in (None, 0):
                return a / b
        return None
    for pattern, repl in [
        (r"[-+]?\d{1,3}(?:\.\d{3})+(?:,\d+)?", lambda m: m.group().replace(".","") .replace(",",".")),
        (r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", lambda m: m.group().replace(",","")),
        (r"[-+]?\d+,\d{3}(?!\d)", lambda m: m.group().replace(",","")),
        (r"[-+]?\d+,\d+", lambda m: m.group().replace(",",".") if len(m.group().split(",")[-1]) != 3 else m.group().replace(",","")),
    ]:
        if re.fullmatch(pattern, text):
            text = repl(re.match(pattern, text))
            break
    try:
        out = float(text)
    except ValueError:
        return None
    return out if math.isfinite(out) else None

def parse_number(text):
    """Parse CHỈ KHI toan bo answer la plain numeric."""
    if text is None: return None
    text = str(text).strip()
    if not text or re.search(r"[\\{}^_a-zA-Z]", text):
        return None
    return _parse_number_core(text) if re.fullmatch(PLAIN_NUMERIC_RE, text) else None


# ── Answer normalization ───────────────────────────────────────────────────────
def _norm_component(token):
    token = str(token or "").strip()
    for pat, repl in [
        (r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", lambda m: m.group().replace(",","")),
        (r"[-+]?\d{1,3}(?:\.\d{3})+(?:,\d+)?", lambda m: m.group().replace(".","").replace(",",".")),
        (r"[-+]?\d+,\d+", lambda m: m.group().replace(",",".")),
    ]:
        if re.fullmatch(pat, token):
            return repl(re.fullmatch(pat, token))
    return token

def normalize_multi_value(answer):
    """Normalize tuple/list rõ rang. Trả về None nếu khong phải."""
    answer = str(answer or "").strip()
    m = re.fullmatch(r"([\[\(])\s*(.+?)\s*([\]\)])", answer)
    if m:
        left, inner, right = m.group(1), m.group(2), m.group(3)
        if re.search(r",\s+", inner):
            parts = re.split(r",\s+", inner)
        elif re.fullmatch(r"[-+]?\d+\s*,\s*[-+]?\d+", inner):
            parts = re.split(r"\s*,\s*", inner)
        else:
            return None
        if len(parts) < 2: return None
        return left + ", ".join(_norm_component(p) for p in parts) + right
    if re.search(r",\s+", answer):
        parts = re.split(r",\s+", answer)
        if len(parts) >= 2:
            return ", ".join(_norm_component(p) for p in parts)
    return None

def is_bad_format(answer):
    answer = str(answer or "").strip()
    if not answer:
        return True
    if re.search(r"\d+\.\d+\.\d+", answer):
        return True
    if re.search(r"\d,\d,\d", answer):
        return True
    if answer.count("(") != answer.count(")"):
        return True
    if answer.count("[") != answer.count("]"):
        return True
    if answer.count("{") != answer.count("}"):
        return True
    return False

def normalize_answer(answer):
    """Preserve LaTeX/tuple, normalize plain numeric."""
    answer = clean_answer_tail(answer) or ""
    answer = re.sub(r"\s+", " ", answer).strip()
    if not answer: return answer
    multi = normalize_multi_value(answer)
    if multi is not None: return multi.strip(" .。;；,，")
    if re.search(r"[\\{}^_]", answer): return answer.strip(" .。;；,，")
    for pat, repl in [
        (r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", lambda m: m.group().replace(",","")),
        (r"[-+]?\d{1,3}(?:\.\d{3})+(?:,\d+)?", lambda m: m.group().replace(".","").replace(",",".")),
        (r"[-+]?\d+,\d+", lambda m: m.group().replace(",",".")),
    ]:
        if re.fullmatch(pat, answer):
            return repl(re.fullmatch(pat, answer))
    answer = re.sub(
        r"^([-+]?\d+(?:\.\d+)?(?:\s*/\s*[-+]?\d+(?:\.\d+)?)?)(?:\s+[a-zA-Z\u00c0-\u1ef9%].*)$",
        r"\1", answer)
    return answer.strip(" .。;；,，")


# ── Filters ────────────────────────────────────────────────────────────────────
def is_english_dominant(text):
    text = normalize_space(text).lower()
    if word_count(text) < 6: return False
    words = re.findall(r"[a-zA-ZÀ-ỹ]+", text)
    if not words: return False
    en_h = sum(1 for w in words if w in EN_HINT)
    vi_h = sum(1 for w in words if w in VI_HINT)
    has_vi_diacritic = bool(VI_DIACRITIC_RE.search(text))
    ascii_words = re.findall(r"\b[a-zA-Z]{2,}\b", text)
    ascii_ratio = len(ascii_words) / max(1, len(words))
    if not has_vi_diacritic and ascii_ratio >= 0.75 and en_h >= 3 and vi_h == 0: return True
    if en_h >= 5 and en_h >= 2 * max(1, vi_h): return True
    return False

def is_answer_only(text):
    return bool(re.fullmatch(
        r"Đáp\s*án\s*là\s*[:：]?\s*[^\n]+",
        str(text or "").strip(),
        flags=re.IGNORECASE,
    ))

def apply_target_mode(response, answer):
    if TARGET_MODE == "answer_only":
        return f"Đáp án là: {answer}", None
    if TARGET_MODE == "solution" and is_answer_only(response):
        return None, "answer_only_response"
    return response, None


# ── Core preprocessing ────────────────────────────────────────────────────────
def preprocess_record(rec, idx):
    query_raw = fix_artifacts(rec.get("query_vi", ""))
    response_raw = fix_artifacts(rec.get("response_vi", ""))
    if not query_raw or not response_raw:
        return None, "missing_query_or_response"

    query = clean_text(strip_asy(query_raw))
    response = clean_text(strip_asy(response_raw))

    if DROP_ENGLISH_DOMINANT and (is_english_dominant(query) or is_english_dominant(response)):
        return None, "english_dominant"
    if len(query) > MAX_QUERY_CHARS or word_count(query) > MAX_QUERY_WORDS:
        return None, "query_too_long"

    answer = normalize_answer(extract_answer(response))
    if DROP_BAD_ANSWER_FORMAT and is_bad_format(answer):
        return None, "bad_answer_format"
    if not answer and DROP_TRAIN_WITHOUT_FINAL_ANSWER:
        return None, "extract_failed"

    if answer:
        response = rebuild_response(response, answer)
        response, reason = apply_target_mode(response, answer)
        if reason: return None, reason

    return {
        "id": rec.get("id", idx),
        "query_vi": query, "response_vi": response,
        "type": rec.get("type", "unknown"),
        "answer_text": answer or None,
        "answer_num": parse_number(answer) if answer else None,
        "target_mode": TARGET_MODE,
    }, None

def process_train(records):
    kept, drop_reasons, failed_samples, asy_count = [], [], [], 0
    for i, rec in enumerate(records):
        had_asy = "[asy]" in (rec.get("query_vi","")+"\n"+rec.get("response_vi","")).lower()
        item, reason = preprocess_record(rec, i)
        if reason:
            drop_reasons.append(reason)
            if reason == "extract_failed" and len(failed_samples) < 50:
                failed_samples.append({"index":i,"type":rec.get("type","unknown"),
                    "query_vi":normalize_space(rec.get("query_vi",""))[:180],
                    "response_tail":str(rec.get("response_vi",""))[-300:]})
        else:
            if had_asy: asy_count += 1
            kept.append(item)
    return kept, Counter(drop_reasons), {"extract_failed_preview":failed_samples,"asy_stripped_count":asy_count}

def process_eval_or_test(records, has_response):
    out = []
    for i, rec in enumerate(records):
        query = clean_text(strip_asy(fix_artifacts(rec.get("query_vi",""))))
        if not query or len(query) > MAX_QUERY_CHARS or word_count(query) > MAX_QUERY_WORDS:
            continue
        item = {"id": rec.get("id", i), "query_vi": query, "type": rec.get("type","unknown")}
        if has_response:
            response = clean_text(strip_asy(fix_artifacts(rec.get("response_vi",""))))
            answer = normalize_answer(extract_answer(response))
            if answer: response = rebuild_response(response, answer)
            item.update({"response_vi": response, "answer_text": answer or None,
                         "answer_num": parse_number(answer) if answer else None, "target_mode": TARGET_MODE})
        out.append(item)
    return out


# ── Dedup ──────────────────────────────────────────────────────────────────────
def dedup_exact(records):
    seen, kept, dup = set(), [], 0
    for rec in records:
        key = (normalize_space(rec.get("query_vi","")).lower(),
               normalize_space(rec.get("response_vi","")).lower())
        if key in seen: dup += 1
        else: seen.add(key); kept.append(rec)
    return kept, dup

def dedup_shortest_same_answer(records):
    groups = defaultdict(list)
    for rec in records:
        q = normalize_space(rec.get("query_vi","")).lower()
        a = normalize_space(rec.get("answer_text","")).lower()
        groups[(q, a)].append(rec)
    kept, dropped = [], 0
    for items in groups.values():
        if len(items) == 1:
            kept.append(items[0])
        else:
            best = min(items, key=lambda r: word_count(str(r.get("response_vi","")).strip()))
            dropped += len(items) - 1
            kept.append(best)
    return kept, dropped


# ── Quality & conflict ─────────────────────────────────────────────────────────
def quality_score(rec):
    score, ans_num, ans_text = 0.0, rec.get("answer_num"), str(rec.get("answer_text") or "")
    if ans_num is not None:
        score += 100 if (ans_num == int(ans_num) and abs(ans_num) < 1e9) else (
            60 if re.fullmatch(r"[-+]?\d+\.\d{1,2}", ans_text) else 40)
    else:
        score += 40 if re.search(r"\\frac|\\sqrt|\\boxed|[_^]", ans_text) else (
            20 if re.fullmatch(r"[a-zA-Z]", ans_text) else -30)
    w = rec.get("total_words", 0)
    score += 70 if w <= 150 else 50 if w <= 250 else 25 if w <= 400 else 0
    score += 40 if not rec.get("was_truncated") else 0
    score += TYPE_BONUS.get(rec.get("type","unknown"), 0)
    return score

def drop_conflicts(records):
    groups = defaultdict(list)
    for r in records:
        groups[normalize_space(r.get("query_vi","")).lower()].append(r)
    kept, report = [], {"conflict_groups":0,"dropped":0,"samples":[]}
    for items in groups.values():
        answers = {normalize_space(x.get("answer_text","")).lower() for x in items}
        if len(answers) > 1:
            report["conflict_groups"] += 1
            if DROP_CONFLICT_GROUPS:
                report["dropped"] += len(items)
                if len(report["samples"]) < 20:
                    report["samples"].append({"query":list(items)[0].get("query_vi","")[:100],
                        "n":len(items),"answers":sorted(answers)[:5]})
            else:
                best = max(items, key=lambda x: x.get("_quality_score", 0))
                kept.append(best); report["dropped"] += len(items) - 1
                if len(report["samples"]) < 20:
                    report["samples"].append({"query":best.get("query_vi","")[:100],
                        "best_score":best.get("_quality_score", 0)})
        else:
            kept.extend(items)
    return kept, report


# ── Arithmetic filter ──────────────────────────────────────────────────────────
def arith_ok(rec):
    text = str(rec.get("response_vi","")); passed = failed = 0
    for m in list(ARITH_EQ_RE.finditer(text))[:10]:
        try:
            a = float(m.group(1).replace(",",".")); b = float(m.group(3).replace(",","."))
            c = float(m.group(4).replace(",",".")); op = m.group(2)
        except ValueError: continue
        exp = {"+":a+b,"-":a-b,"*":a*b,"×":a*b}.get(op, None if abs(b)<1e-12 else a/b)
        if exp is None: continue
        if abs(exp - c) <= max(1e-4, 1e-4*max(1,abs(exp),abs(c))): passed += 1
        else: failed += 1
    t = passed + failed
    return t == 0 or (failed / t) <= ARITH_MAX_FAIL_RATIO


# ── Truncation ─────────────────────────────────────────────────────────────────
def split_response(response, answer):
    response = str(response or "").rstrip()
    m = re.search(r"\nĐáp án là:\s*([^\n]+)\s*$", response, flags=re.IGNORECASE)
    if m: return response[:m.start()].rstrip(), "\nĐáp án là: " + m.group(1).strip()
    suffix = "\nĐáp án là: " + str(answer or extract_answer(response, True) or "").strip()
    body = re.sub(r"\n?Đáp án là:\s*[^\n]+\s*$", "", response, flags=re.IGNORECASE).rstrip()
    return body, suffix

def truncate_record(rec, max_words):
    body, suffix = split_response(rec.get("response_vi",""), rec.get("answer_text"))
    body_words, suffix_words = word_count(body), word_count(suffix)
    total_resp = body_words + suffix_words
    item = dict(rec); item["was_truncated"] = False; item["truncation_strategy"] = "none"
    if total_resp <= max_words:
        item["response_words"] = total_resp
        item["total_words"] = word_count(rec.get("query_vi","")) + total_resp
        return item, None
    budget = max_words - suffix_words
    if budget <= 0: return None, "too_long_prompt_or_answer"
    head_ratio = 0.70 if rec.get("type","") in {"MATH_SV","MATH_FOBAR"} else 0.45
    head_words = max(1, int(budget * head_ratio))
    tail_words = max(1, budget - head_words)
    parts = body.split()
    if len(parts) > budget:
        new_body = (" ".join(parts[:head_words]) + TRUNCATION_MARKER +
                    " ".join(parts[-tail_words:])).strip()
    else:
        new_body = body
    new_resp = (new_body.rstrip() + suffix) if new_body else suffix.lstrip()
    new_total = word_count(rec.get("query_vi","")) + word_count(new_resp)
    item.update({"response_vi": new_resp, "response_words": word_count(new_resp),
                 "was_truncated": True, "truncation_strategy": "head_tail_keep_answer_suffix",
                 "total_words": new_total})
    return item, None

def apply_truncation(records, max_words):
    kept, counter, examples = [], Counter(), []
    for rec in records:
        item, reason = truncate_record(rec, max_words)
        if reason:
            counter[reason] += 1
            if len(examples) < 20:
                examples.append({"id":rec.get("id"),"type":rec.get("type"),
                                 "reason":reason,"query_vi":rec.get("query_vi","")[:180]})
        else:
            if item.get("was_truncated"): counter["smart_truncated"] += 1
            kept.append(item)
    return kept, counter, examples

def percentile(values, p):
    if not values: return None
    values = sorted(values)
    k = (len(values) - 1) * (p / 100)
    f = math.floor(k)
    c = math.ceil(k)
    if f == c: return values[int(k)]
    return values[f] * (c - k) + values[c] * (k - f)

def type_counter(records):
    return Counter(str(r.get("type", "unknown")) for r in records)

def summarize_token_by_type(records):
    groups = defaultdict(list)
    for r in records:
        t = str(r.get("type", "unknown"))
        if r.get("train_tokens") is not None:
            groups[t].append(int(r["train_tokens"]))
    out = {}
    for t, vals in groups.items():
        out[t] = {
            "count": len(vals),
            "min": min(vals),
            "median": round(percentile(vals, 50), 2),
            "p90": round(percentile(vals, 90), 2),
            "p95": round(percentile(vals, 95), 2),
            "max": max(vals),
        }
    return dict(sorted(out.items()))


def summarize_over_limit_by_type(records, max_tokens):
    total_by_type = Counter()
    over_by_type = Counter()
    for r in records:
        t = str(r.get("type", "unknown"))
        total_by_type[t] += 1
        if int(r.get("train_tokens", 0)) > max_tokens:
            over_by_type[t] += 1
    rows = {}
    for t in sorted(total_by_type):
        total = total_by_type[t]
        over = over_by_type[t]
        keep = total - over
        rows[t] = {
            "total": total,
            "over_limit": over,
            "kept_if_drop": keep,
            "over_ratio": round(over / max(1, total), 4),
            "kept_ratio": round(keep / max(1, total), 4),
        }
    return rows

def audit_token_lengths(records, split_name="train"):
    """
    Audit token length sau preprocessing.
    Đo:
      - prompt_tokens
      - response_tokens
      - train_tokens = prompt + response + eos
    Có report theo type để tránh drop lệch các nhóm ít mẫu.
    """
    report = {
        "enabled": TOKEN_AUDIT_ENABLED,
        "split": split_name,
        "model_dir": str(MODEL_DIR),
        "max_train_tokens": MAX_TRAIN_TOKENS,
        "drop_over_limit": TOKEN_AUDIT_DROP_OVER_LIMIT,
        "protect_rare_types": TOKEN_DROP_PROTECT_RARE_TYPES,
        "min_keep_per_type": TOKEN_MIN_KEEP_PER_TYPE,
        "tokenizer_loaded": False,
        "before": len(records),
        "after": len(records),
        "before_by_type": dict(type_counter(records)),
        "after_by_type": dict(type_counter(records)),
        "over_limit": 0,
        "over_limit_ratio": 0.0,
        "stats": {},
        "token_stats_by_type": {},
        "over_limit_by_type": {},
        "over_limit_preview": [],
    }

    if not TOKEN_AUDIT_ENABLED:
        return records, report

    tok = load_audit_tokenizer()
    if tok is None:
        return records, report

    report["tokenizer_loaded"] = True

    audited = []
    train_lens = []
    prompt_lens = []
    response_lens = []
    over_preview = []

    eos_id = getattr(tok, "eos_token_id", None)

    for rec in records:
        prompt = build_train_prompt(rec)
        response = str(rec.get("response_vi", "")).strip()

        prompt_ids = tok(prompt, add_special_tokens=False)["input_ids"]
        response_ids = tok(response, add_special_tokens=False)["input_ids"]

        if eos_id is not None:
            response_ids = response_ids + [eos_id]

        prompt_tokens = len(prompt_ids)
        response_tokens = len(response_ids)
        train_tokens = prompt_tokens + response_tokens

        item = dict(rec)
        item["prompt_tokens"] = prompt_tokens
        item["response_tokens"] = response_tokens
        item["train_tokens"] = train_tokens
        item["over_train_token_limit"] = train_tokens > MAX_TRAIN_TOKENS

        audited.append(item)

        train_lens.append(train_tokens)
        prompt_lens.append(prompt_tokens)
        response_lens.append(response_tokens)

        if train_tokens > MAX_TRAIN_TOKENS and len(over_preview) < TOKEN_AUDIT_PREVIEW_LIMIT:
            over_preview.append({
                "id": item.get("id"),
                "type": item.get("type"),
                "train_tokens": train_tokens,
                "prompt_tokens": prompt_tokens,
                "response_tokens": response_tokens,
                "query_words": item.get("query_words"),
                "response_words": item.get("response_words"),
                "query_vi": str(item.get("query_vi", ""))[:160],
                "answer_text": item.get("answer_text"),
            })

    over_count = sum(1 for x in train_lens if x > MAX_TRAIN_TOKENS)

    report["over_limit"] = over_count
    report["over_limit_ratio"] = round(over_count / max(1, len(audited)), 4)
    report["over_limit_preview"] = over_preview
    report["token_stats_by_type"] = summarize_token_by_type(audited)
    report["over_limit_by_type"] = summarize_over_limit_by_type(audited, MAX_TRAIN_TOKENS)

    if train_lens:
        report["stats"] = {
            "train_tokens": {
                "min": int(min(train_lens)),
                "median": round(percentile(train_lens, 50), 2),
                "p90": round(percentile(train_lens, 90), 2),
                "p95": round(percentile(train_lens, 95), 2),
                "p99": round(percentile(train_lens, 99), 2),
                "max": int(max(train_lens)),
            },
            "prompt_tokens": {
                "min": int(min(prompt_lens)),
                "median": round(percentile(prompt_lens, 50), 2),
                "p95": round(percentile(prompt_lens, 95), 2),
                "max": int(max(prompt_lens)),
            },
            "response_tokens": {
                "min": int(min(response_lens)),
                "median": round(percentile(response_lens, 50), 2),
                "p95": round(percentile(response_lens, 95), 2),
                "max": int(max(response_lens)),
            },
        }

    # Nếu chỉ audit thì trả toàn bộ audited, không drop.
    if not TOKEN_AUDIT_DROP_OVER_LIMIT:
        report["after"] = len(audited)
        report["after_by_type"] = dict(type_counter(audited))
        return audited, report

    # Drop cơ bản: giữ mẫu <= max token.
    kept = [r for r in audited if not r.get("over_train_token_limit")]

    # Bảo vệ type hiếm nếu cần.
    if TOKEN_DROP_PROTECT_RARE_TYPES:
        kept_by_type = type_counter(kept)
        audited_by_type = defaultdict(list)

        for r in audited:
            audited_by_type[str(r.get("type", "unknown"))].append(r)

        extra_keep = []

        for t, items in audited_by_type.items():
            current = kept_by_type.get(t, 0)

            if current >= TOKEN_MIN_KEEP_PER_TYPE:
                continue

            need = TOKEN_MIN_KEEP_PER_TYPE - current

            # Chỉ lấy từ các mẫu over-limit của type đó.
            candidates = [
                r for r in items
                if r.get("over_train_token_limit")
            ]

            # Ưu tiên mẫu vượt ít nhất, tức train_tokens thấp hơn.
            candidates = sorted(candidates, key=lambda r: int(r.get("train_tokens", 10**9)))

            extra_keep.extend(candidates[:need])

        if extra_keep:
            existed = {id(r) for r in kept}
            for r in extra_keep:
                if id(r) not in existed:
                    r = dict(r)
                    r["kept_by_rare_type_protection"] = True
                    kept.append(r)

    report["after"] = len(kept)
    report["after_by_type"] = dict(type_counter(kept))
    report["dropped_by_token"] = len(audited) - len(kept)

    return kept, report
    

# ── Pipeline ──────────────────────────────────────────────────────────────────
def full_pipeline(raw_train, raw_valid=None, raw_test=None):
    print(f"Config: TARGET_MODE={TARGET_MODE}, DROP_CONFLICT_GROUPS={DROP_CONFLICT_GROUPS}, "
          f"DROP_ENGLISH_DOMINANT={DROP_ENGLISH_DOMINANT}, DROP_BAD_ANSWER_FORMAT={DROP_BAD_ANSWER_FORMAT}, "
          f"ARITH_FILTER_ENABLED={ARITH_FILTER_ENABLED}")

    # 1. Text preprocess
    train_records, drop_cnt, pp_logs = process_train(raw_train)
    n1 = len(train_records)
    print(f"[1/7] Text preprocess: {len(raw_train)} -> {n1} (dropped: {sum(drop_cnt.values())})")

    # 2a. Dedup exact
    train_records, n_dup = dedup_exact(train_records)
    n2 = len(train_records)
    print(f"[2/7] Dedup exact: removed {n_dup}, remaining {n2}")

    # 2b. Dedup shortest same answer
    train_records, n_short = dedup_shortest_same_answer(train_records)
    n3 = len(train_records)
    print(f"    Dedup shortest (same query+answer): removed {n_short}, remaining {n3}")

    # 3. Word count + quality score
    print(f"[3/7] Word count + quality score cho {n3} records...")
    for rec in train_records:
        qw = word_count(rec.get("query_vi",""))
        rw = word_count(str(rec.get("response_vi","")).strip())
        rec["query_words"] = qw
        rec["response_words"] = rw
        rec["total_words"] = qw + rw
        rec["was_truncated"] = False
        rec["_quality_score"] = quality_score(rec)
    dist = [r["total_words"] for r in train_records]
    print(f"     Words: min={min(dist)}, median={np.median(dist):.0f}, "
          f"p95={np.percentile(dist,95):.0f}, max={max(dist)}")

    # 4. Conflict resolution
    train_records, conflict_report = drop_conflicts(train_records)
    n4 = len(train_records)
    print(f"[4/7] Conflict ({'drop_groups' if DROP_CONFLICT_GROUPS else 'keep_best'}): "
          f"dropped {conflict_report['dropped']}, con {n4}")

    # 5. Truncation
    train_records, trunc_cnt, trunc_examples = apply_truncation(train_records, MAX_RESPONSE_WORDS)
    n5 = len(train_records)
    print(f"[5/7] Truncation: con {n5}. "
          f"Truncated={trunc_cnt.get('smart_truncated',0)}, "
          f"dropped={trunc_cnt.get('too_long_prompt_or_answer',0)}")

    # 6. Filters
    n6 = len(train_records)
    train_records = [r for r in train_records
                      if QUALITY_FILTER_MIN_WORDS <= r.get("total_words",0) <= QUALITY_FILTER_MAX_WORDS]
    n7 = len(train_records)
    print(f"[6/7] Word filter [{QUALITY_FILTER_MIN_WORDS}..{QUALITY_FILTER_MAX_WORDS}]: {n6} -> {n7}")
    if ARITH_FILTER_ENABLED:
        before = len(train_records)
        train_records = [r for r in train_records if arith_ok(r)]
        print(f"    Arithmetic filter: {before} -> {len(train_records)}")

    # 7. Token audit
    before_token_audit = len(train_records)
    train_records, token_audit_report = audit_token_lengths(train_records, split_name="train")
    after_token_audit = len(train_records)
    def print_over_limit_by_type(token_audit_report):
        rows = token_audit_report.get("over_limit_by_type", {})
        if not rows:
            return

        print("     Over limit by type:")
        print("     type              total   over   kept_if_drop   over_ratio")

        for t, s in sorted(rows.items(), key=lambda kv: kv[1]["over_ratio"], reverse=True):
            print(
                f"     {t:<16} "
                f"{s['total']:>6} "
                f"{s['over_limit']:>6} "
                f"{s['kept_if_drop']:>12} "
                f"{s['over_ratio']:>10.4f}"
            )

    if token_audit_report.get("tokenizer_loaded"):
        stats = token_audit_report.get("stats", {}).get("train_tokens", {})
        print(
            f"[7/7] Token audit <= {MAX_TRAIN_TOKENS}: "
            f"{before_token_audit} -> {after_token_audit}. "
            f"Over limit: {token_audit_report.get('over_limit', 0)} "
            f"({token_audit_report.get('over_limit_ratio', 0.0)})"
        )
        print(
            f"     Train tokens: min={stats.get('min')}, "
            f"median={stats.get('median')}, "
            f"p95={stats.get('p95')}, max={stats.get('max')}"
        )
        print_over_limit_by_type(token_audit_report)
    else:
        print("[7/7] Token audit skipped")

    n_final = len(train_records)

    valid_records = process_eval_or_test(raw_valid, True) if raw_valid else []
    test_records = process_eval_or_test(raw_test, False) if raw_test else []
    print(f"    Valid: {len(valid_records)}, Test: {len(test_records)}")

    report = {
        "train_before": len(raw_train),
        "train_after_text_preprocessing": n1,
        "train_after_exact_dedup": n2,
        "train_after_shortest_same_answer": n3,
        "train_after_conflict": n4,
        "train_after_truncation": n5,
        "train_after_word_filter": n7,
        "train_final": n_final,
        "dropped_text_preprocessing": sum(drop_cnt.values()),
        "drop_reasons": dict(drop_cnt),
        "n_duplicates_exact": n_dup,
        "n_same_answer_dropped": n_short,
        "conflict_report": conflict_report,
        "valid": len(valid_records),
        "test": len(test_records),
        "fingerprint": fingerprint(train_records),
        "smart_truncated": int(trunc_cnt.get("smart_truncated",0)),
        "dropped_too_long": int(trunc_cnt.get("too_long_prompt_or_answer",0)),
        "truncation_drop_preview": trunc_examples,
        "arith_filter_enabled": ARITH_FILTER_ENABLED,
        "target_mode": TARGET_MODE,
        "drop_conflict_groups": DROP_CONFLICT_GROUPS,
        "drop_english_dominant": DROP_ENGLISH_DOMINANT,
        "drop_bad_answer_format": DROP_BAD_ANSWER_FORMAT,
        "token_audit": token_audit_report,
        "train_after_token_audit": n_final,
        **pp_logs,
    }
    return train_records, valid_records, test_records, report

## Cell 4 — Preprocess to answer-only records

In [5]:
# ============================================================
# Full preprocessing pipeline from preprocessing_pipeline_clean(3).py
# TARGET_MODE = "answer_only"
# ============================================================

train_records_pp, valid_records_pp, test_records_pp, prep_report = full_pipeline(
    raw_train,
    raw_valid,
    raw_test
)

# Convert full_pipeline output back to the notebook's answer-only schema.
# This lets Cell 10/18/20 keep using prompt/target/final_answer.

def normalize_query(q: str) -> str:
    return clean_text(strip_asy(fix_artifacts(str(q or ""))))


def make_prompt(query: str) -> str:
    return f"Câu hỏi: {normalize_query(query)}\nĐáp án:"


def make_target(answer_or_response: str) -> str:
    s = str(answer_or_response or "").strip()
    ans = normalize_answer(extract_answer(s, allow_last=True)) or s
    ans = re.sub(r"^Đáp\s*án\s*là\s*[:：]?\s*", "", str(ans), flags=re.IGNORECASE).strip()
    return f" {ans}"


def adapt_answer_only_records(records: List[Dict[str, Any]], split: str) -> List[Dict[str, Any]]:
    out = []
    for i, r in enumerate(records):
        ans = r.get("answer_text")
        response = str(r.get("response_vi", "") or "").strip()

        # In TARGET_MODE="answer_only", response_vi should already be "Đáp án là: {answer}".
        target = make_target(response if response else ans)

        item = dict(r)
        item.update({
            "raw_index": r.get("raw_index", i),
            "final_answer": str(ans or "").strip(),
            "prompt": make_prompt(r.get("query_vi", "")),
            "target": target,
        })
        out.append(item)
    return out


train_records = adapt_answer_only_records(train_records_pp, "train")
valid_records = adapt_answer_only_records(valid_records_pp, "valid")
test_records = test_records_pp

# Shuffle and cap train set after full preprocessing.
rng = random.Random(SEED)
rng.shuffle(train_records)

train_before_cap = len(train_records)
if TRAIN_MAX_SAMPLES is not None and len(train_records) > TRAIN_MAX_SAMPLES:
    train_records = train_records[:TRAIN_MAX_SAMPLES]

prep_report["variant"] = VARIANT_NAME
prep_report["target_format"] = "answer_only_with_full_preprocessing"
prep_report["train_before_cap"] = train_before_cap
prep_report["train_used_for_training"] = len(train_records)
prep_report["max_length_notebook"] = MAX_LENGTH
prep_report["max_train_tokens_preprocess"] = MAX_TRAIN_TOKENS
prep_report["token_min_keep_per_type"] = TOKEN_MIN_KEEP_PER_TYPE

write_jsonl(train_records, PROC_DIR / "train_answer_only_full_preprocessed.jsonl")
write_jsonl(valid_records, PROC_DIR / "valid_answer_only_full_preprocessed.jsonl")
write_json(prep_report, PROC_DIR / "preprocess_report.json")

print(json.dumps({
    "raw_train": len(raw_train),
    "raw_valid": len(raw_valid),
    "train_after_full_preprocess": len(train_records_pp),
    "valid_after_full_preprocess": len(valid_records_pp),
    "train_used_for_training": len(train_records),
    "target_mode": TARGET_MODE,
    "max_length": MAX_LENGTH,
    "max_train_tokens": MAX_TRAIN_TOKENS,
    "token_drop_over_limit": TOKEN_AUDIT_DROP_OVER_LIMIT,
    "protect_rare_types": TOKEN_DROP_PROTECT_RARE_TYPES,
    "min_keep_per_type": TOKEN_MIN_KEEP_PER_TYPE,
    "token_over_limit": prep_report.get("token_audit", {}).get("over_limit"),
    "token_over_limit_ratio": prep_report.get("token_audit", {}).get("over_limit_ratio"),
}, ensure_ascii=False, indent=2))

pd.DataFrame(train_records[:5])[[
    "type",
    "query_vi",
    "answer_text",
    "final_answer",
    "prompt",
    "target",
    # "train_tokens",
    # "over_train_token_limit"
]]

Config: TARGET_MODE=solution, DROP_CONFLICT_GROUPS=True, DROP_ENGLISH_DOMINANT=True, DROP_BAD_ANSWER_FORMAT=True, ARITH_FILTER_ENABLED=False
[1/7] Text preprocess: 95400 -> 94227 (dropped: 1173)
[2/7] Dedup exact: removed 124, remaining 94103
    Dedup shortest (same query+answer): removed 38093, remaining 56010
[3/7] Word count + quality score cho 56010 records...
     Words: min=7, median=155, p95=320, max=845
[4/7] Conflict (drop_groups): dropped 354, con 55656
[5/7] Truncation: con 55656. Truncated=52, dropped=0
[6/7] Word filter [8..450]: 55656 -> 55322
[7/7] Token audit <= 512: 55322 -> 54299. Over limit: 1023 (0.0185)
     Train tokens: min=37, median=205.0, p95=421.0, max=1453
     Over limit by type:
     type              total   over   kept_if_drop   over_ratio
     MATH_FOBAR         2141    449         1692     0.2097
     MATH_SV            2132    182         1950     0.0854
     MATH_Rephrased     8368    188         8180     0.0225
     MATH_AnsAug        4036     90  

,type,query_vi,answer_text,final_answer,prompt,target
0,GSM_Rephrased,Nếu ban đầu Mary có 8 chiếc vít và cần mua gấp...,6,6,Câu hỏi: Nếu ban đầu Mary có 8 chiếc vít và cầ...,6
1,GSM_Rephrased,Nếu có 9 chiếc xe đạp và 16 ô tô trong gara th...,82,82,Câu hỏi: Nếu có 9 chiếc xe đạp và 16 ô tô tron...,82
2,MATH_FOBAR,Tìm $(-1)^{-10} + (-1)^{-9} + (-1)^{-8} + \cdo...,21,21,Câu hỏi: Tìm $(-1)^{-10} + (-1)^{-9} + (-1)^{-...,21
3,MATH_Rephrased,Độ tuổi trung bình của nhóm gồm 33 học sinh lớ...,24.75,24.75,Câu hỏi: Độ tuổi trung bình của nhóm gồm 33 họ...,24.75
4,GSM_FOBAR,Hai vận động viên quyết định thi đấu xem ai có...,34,34,Câu hỏi: Hai vận động viên quyết định thi đấu ...,34


## Cell 5 — Load tokenizer/model and build torch datasets

In [6]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = find_model_path()

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(str(MODEL_PATH), local_files_only=True)

# Force safe eos/pad ids required by the competition note.
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else "<|endoftext|>"
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID

print("vocab_size:", len(tokenizer), "pad:", tokenizer.pad_token_id, "eos:", tokenizer.eos_token_id)

class AnswerOnlyDataset(Dataset):
    def __init__(self, records: List[Dict[str, Any]], tokenizer, max_length: int):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        r = self.records[idx]

        prompt = str(r["prompt"])
        target = str(r["target"]).strip()

        eos = self.tokenizer.eos_token or ""
        target = target + eos

        prompt_ids = self.tokenizer(prompt, add_special_tokens=False)["input_ids"]
        target_ids = self.tokenizer(target, add_special_tokens=False)["input_ids"]

        # Important:
        # Keep target/answer tokens intact.
        # If too long, truncate the prompt side, not the answer side.
        if len(prompt_ids) + len(target_ids) > self.max_length:
            prompt_budget = self.max_length - len(target_ids)

            if prompt_budget <= 0:
                # Extremely rare: target itself is too long.
                # Keep the tail of target so answer ending is preserved.
                prompt_ids = []
                target_ids = target_ids[-self.max_length:]
            else:
                # Keep the tail of the question prompt.
                prompt_ids = prompt_ids[-prompt_budget:]

        input_ids = prompt_ids + target_ids
        attention_mask = [1] * len(input_ids)

        labels = [-100] * len(prompt_ids) + target_ids

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

class CausalLMCollator:
    def __init__(self, tokenizer, label_pad_token_id: int = -100):
        self.tokenizer = tokenizer
        self.label_pad_token_id = label_pad_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch["input_ids"].append(torch.cat([f["input_ids"], torch.full((pad_len,), self.tokenizer.pad_token_id, dtype=torch.long)]))
            batch["attention_mask"].append(torch.cat([f["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
            batch["labels"].append(torch.cat([f["labels"], torch.full((pad_len,), self.label_pad_token_id, dtype=torch.long)]))
        return {k: torch.stack(v) for k, v in batch.items()}

train_dataset = AnswerOnlyDataset(train_records, tokenizer, MAX_LENGTH)
eval_loss_records = valid_records[:min(EVAL_LOSS_MAX_SAMPLES, len(valid_records))]
eval_dataset = AnswerOnlyDataset(eval_loss_records, tokenizer, MAX_LENGTH)
collator = CausalLMCollator(tokenizer)

print("train_dataset:", len(train_dataset), "eval_dataset:", len(eval_dataset))


MODEL_PATH: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab_size: 50258 pad: 50256 eos: 50256
train_dataset: 40000 eval_dataset: 512


## Cell 6 — Apply LoRA

In [7]:

try:
    from peft import LoraConfig, get_peft_model, TaskType
except Exception as e:
    raise ImportError(
        "PEFT is required for this LoRA variant. On Kaggle, attach a dataset/package with peft if needed, still with Internet OFF."
    ) from e

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["c_attn", "c_proj", "c_fc"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available():
    model = model.cuda()


trainable params: 4,718,592 || all params: 129,158,400 || trainable%: 3.6533


## Cell 7 — Fine-tune answer-only LoRA

In [8]:

from transformers import Trainer, TrainingArguments


def build_training_args() -> TrainingArguments:
    raw_kwargs = dict(
        output_dir=str(MODEL_OUT_DIR),
        overwrite_output_dir=True,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        eval_steps=EVAL_STEPS,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        report_to="none",
        dataloader_num_workers=2,
        remove_unused_columns=False,
        load_best_model_at_end=False,
    )
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in sig:
        raw_kwargs["eval_strategy"] = "steps"
    elif "evaluation_strategy" in sig:
        raw_kwargs["evaluation_strategy"] = "steps"

    kwargs = {k: v for k, v in raw_kwargs.items() if k in sig}
    dropped = sorted(set(raw_kwargs) - set(kwargs))
    if dropped:
        print("Dropped unsupported TrainingArguments:", dropped)
    return TrainingArguments(**kwargs)


def build_trainer_kwargs(args: TrainingArguments) -> Dict[str, Any]:
    base = dict(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=collator,
    )
    sig = inspect.signature(Trainer.__init__).parameters
    return {k: v for k, v in base.items() if k in sig}

if DO_TRAIN:
    training_args = build_training_args()
    trainer = Trainer(**build_trainer_kwargs(training_args))
    train_result = trainer.train()
    print(train_result)
    trainer.save_model(str(MODEL_OUT_DIR))
    tokenizer.save_pretrained(str(MODEL_OUT_DIR))
    write_json({"train_result": str(train_result), "variant": VARIANT_NAME}, MODEL_OUT_DIR / "train_result.json")
else:
    print("DO_TRAIN=False, skip training")

model.eval()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Dropped unsupported TrainingArguments: ['overwrite_output_dir']


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
700,2.005402,2.088617
1400,1.842065,1.956714
2100,1.703677,1.907358


TrainOutput(global_step=2500, training_loss=2.045052404785156, metrics={'train_runtime': 3488.8478, 'train_samples_per_second': 45.86, 'train_steps_per_second': 0.717, 'total_flos': 1.0431086520877056e+16, 'train_loss': 2.045052404785156, 'epoch': 4.0})


## Cell 8 — Conservative rule solver

Rule solver chạy trước model. Nó chỉ trả lời khi pattern khá chắc chắn; nếu không chắc thì trả `None` để model xử lý.

In [9]:

def numbers_in_text(text: str) -> List[float]:
    text = normalize_unicode_text(text)
    vals = []
    for m in re.findall(r"-?\d+(?:[\.,]\d+)*", text):
        try:
            vals.append(float(_normalize_num_token(m)))
        except Exception:
            pass
    return vals


def format_number(x: float) -> str:
    if x is None or not math.isfinite(float(x)):
        return ""
    if abs(x - round(x)) < 1e-9:
        return str(int(round(x)))
    s = f"{x:.10f}".rstrip("0").rstrip(".")
    return s


def lcm(a: int, b: int) -> int:
    return abs(a * b) // math.gcd(a, b) if a and b else 0


def count_distinct_prime_factors(n: int) -> int:
    n = abs(int(n))
    cnt = 0
    d = 2
    while d * d <= n:
        if n % d == 0:
            cnt += 1
            while n % d == 0:
                n //= d
        d += 1 if d == 2 else 2
    if n > 1:
        cnt += 1
    return cnt


def safe_eval_arith(expr: str) -> Optional[float]:
    # Only allow a tiny arithmetic subset.
    expr = normalize_unicode_text(expr)
    expr = expr.replace("^", "**")
    if not re.fullmatch(r"[0-9\s\+\-\*\/\.\(\)\*]+", expr):
        return None
    try:
        val = eval(expr, {"__builtins__": {}}, {})
        if isinstance(val, (int, float)) and math.isfinite(float(val)):
            return float(val)
    except Exception:
        return None
    return None


def rule_solver(query: str) -> Tuple[Optional[str], str]:
    q = normalize_query(query).lower()

    # LCM/GCD direct.
    if "bội số chung nhỏ nhất" in q or "lcm" in q:
        nums = [int(x) for x in numbers_in_text(q)]
        if len(nums) >= 2:
            ans = nums[0]
            for n in nums[1:]:
                ans = lcm(ans, n)
            return str(ans), "rule_lcm"

    if "ước chung lớn nhất" in q or "gcd" in q:
        nums = [int(x) for x in numbers_in_text(q)]
        if len(nums) >= 2:
            ans = nums[0]
            for n in nums[1:]:
                ans = math.gcd(ans, n)
            return str(ans), "rule_gcd"

    # Distinct prime factors.
    if "thừa số nguyên tố phân biệt" in q:
        nums = [int(x) for x in numbers_in_text(q)]
        if nums:
            return str(count_distinct_prime_factors(nums[0])), "rule_distinct_prime_factors"

    # floor(sqrt(n))^2.
    if ("căn bậc hai" in q or "sqrt" in q) and ("hàm sàn" in q or "floor" in q):
        nums = numbers_in_text(q)
        if nums:
            v = math.floor(math.sqrt(nums[0]))
            if "bình phương" in q or "^2" in q:
                v = v * v
            return str(int(v)), "rule_floor_sqrt"

    # Board-game / remaining distance: total spaces and already moved.
    m = re.search(r"có\s+(\d+)\s+[^.]{0,80}ô[^.]{0,120}(?:đã đi được|đi được)\s+(\d+)\s+ô", q)
    if m and ("cần" in q or "thêm" in q):
        return str(int(m.group(1)) - int(m.group(2))), "rule_remaining_spaces"

    # Triangle perimeter with two equal sides.
    if "chu vi" in q and "tam giác" in q:
        m = re.search(r"hai cạnh bằng\s+(\d+).*?cạnh còn lại bằng\s+(\d+)", q)
        if m:
            return str(2 * int(m.group(1)) + int(m.group(2))), "rule_triangle_two_equal_sides"
        nums = [int(x) for x in numbers_in_text(q)]
        if len(nums) == 3 and all(n > 0 for n in nums):
            return str(sum(nums)), "rule_triangle_sum_three_sides"

    # Average of explicit daily values with one doubled day: Walmart example pattern.
    if "trung bình" in q and "gấp đôi" in q:
        nums = numbers_in_text(q)
        # Example: 210, 150 => days are 210, 2*210, 150.
        if len(nums) >= 2:
            first = nums[0]
            last = nums[-1]
            return format_number((first + 2 * first + last) / 3), "rule_average_doubled_middle"

    # Simple expression after "tính" if the query is basically arithmetic.
    if q.startswith("tính") or q.startswith("giá trị"):
        exprs = re.findall(r"[-+]?\d+(?:\.\d+)?(?:\s*[\+\-\*\/]\s*[-+]?\d+(?:\.\d+)?)+", q)
        if exprs:
            val = safe_eval_arith(exprs[-1])
            if val is not None:
                return format_number(val), "rule_direct_arithmetic_expression"

    # Unit price / total cost common pattern: each item costs A, number of items B.
    if "mỗi" in q and ("tổng" in q or "bao nhiêu" in q):
        # Very conservative: "mỗi ... có giá X" and "N ..." near same sentence is too broad, so skip unless multiplication phrasing appears.
        pass

    return None, "no_rule"

# Smoke tests.
for q in [
    "Susan đang chơi một trò chơi board game có 48 ô. Sau ba lượt, cô ấy đã đi được 11 ô. Cô ấy cần đi thêm bao nhiêu ô để đến ô cuối?",
    "Một tam giác có hai cạnh bằng 7 và cạnh còn lại bằng 5. Chu vi tam giác là bao nhiêu?",
    "Tìm bội số chung nhỏ nhất của 24 và 90.",
    "56 có bao nhiêu thừa số nguyên tố phân biệt?",
]:
    print(rule_solver(q), "--", q[:80])


(None, 'no_rule') -- Susan đang chơi một trò chơi board game có 48 ô. Sau ba lượt, cô ấy đã đi được 1
('19', 'rule_triangle_two_equal_sides') -- Một tam giác có hai cạnh bằng 7 và cạnh còn lại bằng 5. Chu vi tam giác là bao n
('360', 'rule_lcm') -- Tìm bội số chung nhỏ nhất của 24 và 90.
('2', 'rule_distinct_prime_factors') -- 56 có bao nhiêu thừa số nguyên tố phân biệt?


## Cell 9 — Model generation and final prediction functions

In [10]:

@torch.no_grad()
def generate_answer_only(prompt: str, do_sample: bool = False, num_return_sequences: int = 1) -> List[str]:
    model.eval()
    old_truncation_side = tokenizer.truncation_side
    tokenizer.truncation_side = "left"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)
    tokenizer.truncation_side = old_truncation_side
    gen_kwargs = dict(
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        pad_token_id=SAFE_EOS_ID,
        eos_token_id=SAFE_EOS_ID,
        num_return_sequences=num_return_sequences,
    )
    if do_sample:
        gen_kwargs.update(dict(do_sample=True, temperature=0.7, top_p=0.9))
    else:
        gen_kwargs.update(dict(do_sample=False, num_beams=1))

    out = model.generate(**inputs, **gen_kwargs)
    decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
    answers = []
    for text in decoded:
        # Remove prompt prefix if model repeats it.
        tail = text[len(prompt):] if text.startswith(prompt) else text.split("Đáp án:")[-1]
        ans = extract_final_answer(tail, fallback_last_number=True)
        if ans is None:
            ans = clean_answer_candidate(tail)
        ans = clean_answer_candidate(ans)
        if ans:
            answers.append(ans)
    return answers


def choose_answer_from_candidates(cands: List[str]) -> Tuple[Optional[str], Dict[str, Any]]:
    cleaned = [clean_answer_candidate(c) for c in cands if c and clean_answer_candidate(c)]
    if not cleaned:
        return None, {"source": "none", "candidates": []}
    counts = Counter(cleaned)
    # Prefer most common; tie-break by shorter answer string.
    best = sorted(counts.items(), key=lambda kv: (-kv[1], len(kv[0]), kv[0]))[0][0]
    return best, {"source": "model", "candidates": cleaned, "counts": dict(counts)}


def make_model_output(answer: str, source: str = "model") -> str:
    answer = clean_answer_candidate(answer or "")
    if not answer:
        answer = "0"
    if source.startswith("rule"):
        reason = "Áp dụng quy tắc tính nhanh từ dữ kiện trong đề."
    else:
        reason = "Tính theo dữ kiện trong đề."
    return f"Lời giải ngắn: {reason}\nĐáp án là: {answer}"


def predict_one(query: str) -> Tuple[str, Dict[str, Any]]:
    # 1) Rule solver first.
    rule_ans, rule_name = rule_solver(query)
    if rule_ans is not None:
        return make_model_output(rule_ans, source=rule_name), {"source": rule_name, "answer": rule_ans}

    # 2) Deterministic model.
    prompt = make_prompt(normalize_query(query))
    det_cands = generate_answer_only(prompt, do_sample=False, num_return_sequences=1)
    ans, info = choose_answer_from_candidates(det_cands)
    if ans is not None:
        return make_model_output(ans, source="model_greedy"), {"source": "model_greedy", "answer": ans, **info}

    # 3) Sampling fallback.
    sample_cands = generate_answer_only(prompt, do_sample=True, num_return_sequences=3)
    ans, info = choose_answer_from_candidates(sample_cands)
    if ans is not None:
        return make_model_output(ans, source="model_sample"), {"source": "model_sample", "answer": ans, **info}

    return make_model_output("0", source="fallback_zero"), {"source": "fallback_zero", "answer": "0"}


## Cell 10 — Validation inference + report

In [11]:

def evaluate_records(records: List[Dict[str, Any]], max_samples: Optional[int] = None) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    eval_records = list(records)
    if max_samples is not None and len(eval_records) > max_samples:
        rng = random.Random(SEED)
        eval_records = rng.sample(eval_records, max_samples)

    outputs = []
    for i, r in enumerate(eval_records):
        if i % 50 == 0:
            print(f"Evaluating {i}/{len(eval_records)}")
        model_output, info = predict_one(r["query_vi"])
        pred_answer = extract_final_answer(model_output, fallback_last_number=True)
        gold_answer = r.get("final_answer") or r.get("answer_text")
        sc = score_one(pred_answer, gold_answer)
        err = relative_error(pred_answer, gold_answer)
        outputs.append({
            "id": r.get("id", i),
            "raw_index": r.get("raw_index", i),
            "query_vi": r["query_vi"],
            "type": r["type"],
            "gold_answer": gold_answer,
            "pred_answer": clean_answer_candidate(pred_answer or ""),
            "relative_error": err,
            "score": sc,
            "model_output": model_output,
            "predict_info": info,
        })

    n = len(outputs)
    report = {
        "variant": VARIANT_NAME,
        "n": n,
        "raw_score": int(sum(x["score"] for x in outputs)),
        "score_10": float(sum(x["score"] for x in outputs) / n) if n else 0.0,
        "exact_10_count": int(sum(x["score"] == 10 for x in outputs)),
        "score_5_count": int(sum(x["score"] == 5 for x in outputs)),
        "score_1_count": int(sum(x["score"] == 1 for x in outputs)),
        "score_0_count": int(sum(x["score"] == 0 for x in outputs)),
        "extract_rate": float(np.mean([bool(x["pred_answer"]) for x in outputs])) if n else 0.0,
        "source_counts": dict(Counter(x["predict_info"].get("source", "unknown") for x in outputs)),
    }
    return outputs, report

if DO_VALIDATE:
    valid_outputs, valid_report = evaluate_records(valid_records, VALID_EVAL_MAX_SAMPLES)
    write_json(valid_outputs, WORK_DIR / "valid_output.json")
    write_json(valid_report, WORK_DIR / "valid_report.json")
    print(json.dumps(valid_report, ensure_ascii=False, indent=2))
else:
    valid_outputs, valid_report = [], {}
    print("DO_VALIDATE=False, skip validation")


Evaluating 0/988
Evaluating 50/988
Evaluating 100/988
Evaluating 150/988
Evaluating 200/988
Evaluating 250/988
Evaluating 300/988
Evaluating 350/988
Evaluating 400/988
Evaluating 450/988
Evaluating 500/988
Evaluating 550/988
Evaluating 600/988
Evaluating 650/988
Evaluating 700/988
Evaluating 750/988
Evaluating 800/988
Evaluating 850/988
Evaluating 900/988
Evaluating 950/988
{
  "variant": "v9_solution_full_preprocess_lora_rule_solver",
  "n": 988,
  "raw_score": 1559,
  "score_10": 1.5779352226720649,
  "exact_10_count": 112,
  "score_5_count": 31,
  "score_1_count": 284,
  "score_0_count": 561,
  "extract_rate": 1.0,
  "source_counts": {
    "model": 963,
    "rule_lcm": 4,
    "rule_floor_sqrt": 2,
    "rule_distinct_prime_factors": 1,
    "rule_average_doubled_middle": 3,
    "rule_direct_arithmetic_expression": 11,
    "rule_gcd": 3,
    "rule_triangle_sum_three_sides": 1
  }
}


## Cell 11 — Report by type and error table

In [12]:

if valid_outputs:
    df_eval = pd.DataFrame(valid_outputs)

    def summarize_group(g: pd.DataFrame) -> pd.Series:
        return pd.Series({
            "n": len(g),
            "score_10": g["score"].sum() / len(g) if len(g) else 0,
            "extract_rate": (g["pred_answer"].astype(str).str.len() > 0).mean() if len(g) else 0,
            "score_10_count": int((g["score"] == 10).sum()),
            "score_5_count": int((g["score"] == 5).sum()),
            "score_1_count": int((g["score"] == 1).sum()),
            "score_0_count": int((g["score"] == 0).sum()),
        })

    type_report = df_eval.groupby("type").apply(summarize_group).reset_index()
    type_report = type_report.sort_values(["score_10", "n"], ascending=[True, False])
    display(type_report)
    type_report.to_csv(WORK_DIR / "valid_report_by_type.csv", index=False, encoding="utf-8-sig")

    debug_cols = ["raw_index", "type", "score", "relative_error", "gold_answer", "pred_answer", "query_vi", "model_output", "predict_info"]
    errors_df = df_eval[df_eval["score"] < 10].sort_values(["score", "relative_error"], ascending=[True, False], na_position="last")
    display(errors_df[debug_cols].head(30))
    errors_df[debug_cols].to_csv(WORK_DIR / "valid_errors.csv", index=False, encoding="utf-8-sig")

    full_summary = {"overall": valid_report, "by_type": type_report.to_dict(orient="records"), "preprocess": prep_report}
    write_json(full_summary, WORK_DIR / "valid_full_summary.json")
else:
    print("No valid outputs to report.")


,type,n,score_10,extract_rate,score_10_count,score_5_count,score_1_count,score_0_count
4,MATH_AnsAug,173.0,0.791908,1.0,8.0,3.0,42.0,120.0
0,GSM_AnsAug,208.0,1.052885,1.0,12.0,8.0,59.0,129.0
6,MATH_Rephrased,116.0,1.655172,1.0,14.0,5.0,27.0,70.0
2,GSM_Rephrased,197.0,1.908629,1.0,25.0,11.0,71.0,90.0
3,GSM_SV,95.0,1.978947,1.0,15.0,2.0,28.0,50.0
7,MATH_SV,39.0,2.128205,1.0,7.0,1.0,8.0,23.0
1,GSM_FOBAR,118.0,2.262712,1.0,23.0,0.0,37.0,58.0
5,MATH_FOBAR,42.0,2.309524,1.0,8.0,1.0,12.0,21.0


,raw_index,type,score,relative_error,gold_answer,pred_answer,query_vi,model_output,predict_info
950,950,MATH_AnsAug,0,2.057615e+150,6,"1,2,3,4,5,6,8,9,10,11,12,13,14,15,16,17,18,19,...",Tập hợp các vectơ $\left\{ \begin{pmatrix} 1 \...,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '1,2,3,4,5,6,8,9..."
146,146,GSM_AnsAug,0,1.499000e+03,10,15000,Nancy muốn tìm hiểu xem liệu cô ấy có đủ khả n...,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '15000', 'candid..."
365,365,GSM_FOBAR,0,1.499000e+03,2,3000,Một xe đầu kéo có tải trọng 50000 pound. 10% t...,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '3000', 'candida..."
435,435,GSM_FOBAR,0,9.990000e+02,2,2000,Legacy có 5 thỏi vàng cô nhận được từ cha mình...,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '2000', 'candida..."
932,932,MATH_FOBAR,0,4.070000e+02,0,407,Tính tổng bình phương các nghiệm của phương tr...,Lời giải ngắn: Áp dụng quy tắc tính nhanh từ d...,{'source': 'rule_direct_arithmetic_expression'...
9,9,GSM_AnsAug,0,2.510000e+02,5,1260,Bob được hỗ trợ tiền thuê nhà vì anh ấy có thu...,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '1260', 'candida..."
170,170,MATH_FOBAR,0,9.900000e+01,1,100,"Giả sử $F_n$ là số Fibonacci thứ $n$, trong đó...",Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '100', 'candidat..."
264,264,MATH_SV,0,9.900000e+01,10,1000,Hai dãy số học $A$ và $B$ đều bắt đầu bằng 30 ...,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '1000', 'candida..."
556,556,GSM_AnsAug,0,8.900000e+01,3,270,Johnny đã viết một bài luận khoảng 150 từ. Mad...,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '270', 'candidat..."
438,438,MATH_Rephrased,0,8.400000e+01,3,255,"Cơ số, ký hiệu là b, sao cho số 100 trong cơ s...",Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '255', 'candidat..."


## Cell 12 — Generate `test_predictions.json` when test exists

In [13]:

def build_test_records(raw_test: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return process_eval_or_test(raw_test, has_response=False)


def predict_test(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    preds = []
    for i, r in enumerate(records):
        if i % 50 == 0:
            print(f"Predicting {i}/{len(records)}")
        model_output, info = predict_one(r["query_vi"])
        preds.append({
            "id": r["id"],
            "query_vi": r["query_vi"],
            "type": r["type"],
            "model_output": model_output,
        })
    return preds

if DO_TEST_PREDICT and TEST_PATH is not None and TEST_PATH.exists():
    print("Found test file:", TEST_PATH)
    raw_test = ensure_list_records(read_json_or_jsonl(TEST_PATH))
    test_records = build_test_records(raw_test)
    test_predictions = predict_test(test_records)
    write_json(test_predictions, WORK_DIR / "test_predictions.json")
    print("Saved:", WORK_DIR / "test_predictions.json", "n=", len(test_predictions))
elif DO_TEST_PREDICT:
    # Smoke-test output format using first 2 valid examples so the notebook always leaves an example file.
    print("No official test.json found. Creating sample_test_predictions_from_valid.json for format check only.")
    sample_records = [
        {"id": r["id"], "query_vi": r["query_vi"], "type": r["type"]}
        for r in valid_records[:2]
    ]
    sample_predictions = predict_test(sample_records)
    write_json(sample_predictions, WORK_DIR / "sample_test_predictions_from_valid.json")
    display(pd.DataFrame(sample_predictions))
else:
    print("DO_TEST_PREDICT=False, skip test prediction")


No official test.json found. Creating sample_test_predictions_from_valid.json for format check only.
Predicting 0/2


,id,query_vi,type,model_output
0,0,Nếu Susan đang chơi một trò chơi cờ bàn có 48 ...,GSM_Rephrased,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...
1,1,"Nếu $\angle PQR = \angle PRQ$, và độ dài của Q...",MATH_Rephrased,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...


## Cell 13 — Final artifact list

In [14]:

print("Important outputs:")
for p in [
    PROC_DIR / "train_answer_only_full_preprocessed.jsonl",
    PROC_DIR / "valid_answer_only_full_preprocessed.jsonl",
    PROC_DIR / "preprocess_report.json",
    WORK_DIR / "valid_output.json",
    WORK_DIR / "valid_report.json",
    WORK_DIR / "valid_report_by_type.csv",
    WORK_DIR / "valid_errors.csv",
    WORK_DIR / "test_predictions.json",
    WORK_DIR / "sample_test_predictions_from_valid.json",
    MODEL_OUT_DIR,
]:
    print("-", p, "exists=", p.exists())

Important outputs:
- /kaggle/working/processed_v9_solution_full_preprocess_lora_rule_solver/train_answer_only_full_preprocessed.jsonl exists= True
- /kaggle/working/processed_v9_solution_full_preprocess_lora_rule_solver/valid_answer_only_full_preprocessed.jsonl exists= True
- /kaggle/working/processed_v9_solution_full_preprocess_lora_rule_solver/preprocess_report.json exists= True
- /kaggle/working/valid_output.json exists= True
- /kaggle/working/valid_report.json exists= True
- /kaggle/working/valid_report_by_type.csv exists= True
- /kaggle/working/valid_errors.csv exists= True
- /kaggle/working/test_predictions.json exists= False
- /kaggle/working/sample_test_predictions_from_valid.json exists= True
- /kaggle/working/lora_v9_solution_full_preprocess_lora_rule_solver exists= True
